# 3.0 Post-Training Quantization for a Vision-Language Model

In this notebook you will learn how to:

- Apply FP8 (and optionally NVFP4) post-training quantization to a VLM checkpoint.
- Use ModelOpt's existing [`examples/llm_ptq/hf_ptq.py`](../../llm_ptq/hf_ptq.py) flow — no Megatron-Bridge required for this step.

Prerequisite: notebooks [`01_pruning_vlm.ipynb`](01_pruning_vlm.ipynb) and [`02_distillation_vlm.ipynb`](02_distillation_vlm.ipynb) have produced a pruned + distilled VLM. Quantization runs as a separate, final stage on the resulting HF checkpoint.

---
## 3.1 Why quantization is separate from prune/distill

Pruning and distillation reshape the *architecture* (fewer layers, narrower FFN, smaller attention) — they have to happen in BF16/FP16 because the optimizer needs full-precision gradients. Quantization is a one-shot calibration step on the final weights: it converts FP/BF16 tensors into a low-bit format (FP8, NVFP4, INT8, …) without retraining. So the right order is **prune → distill → quantize**.

ModelOpt's PTQ pipeline already understands VLM checkpoints: `hf_ptq.py` calls `is_multimodal_model` to detect the VLM, walks to `model.language_model`, quantizes only the language tower, and writes a unified HF checkpoint with the vision encoder and projector left in BF16. The `--calib_with_images` flag lets calibration see image-text pairs so activation ranges in the projector → LM interface reflect real multimodal traffic.

---
## 3.2 FP8 PTQ

FP8 is the safe default: H100/B200 natively run FP8 matmuls, and the quality drop is usually under 1% on Qwen-VL-class models.

In [1]:
!nvidia-smi

Mon May 18 09:52:41 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 570.195.03             Driver Version: 570.195.03     CUDA Version: 13.1     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA H100 80GB HBM3          On  |   00000000:0B:00.0 Off |                    0 |
| N/A   32C    P0             69W /  700W |       0MiB /  81559MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [4]:
DISTILLED_PATH = "/workspace/user_homes/lmikaelyan/Qwen3-VL-8B-Instruct" #"/workspace/user_homes/lmikaelyan/Qwen3-VL-distilled-smoke"
EXPORT_PATH = "/workspace/user_homes/lmikaelyan/Qwen3-VL-distilled-fp8"

!python ../../llm_ptq/hf_ptq.py \
    --pyt_ckpt_path {DISTILLED_PATH} \
    --export_path   {EXPORT_PATH} \
    --qformat fp8 \
    --kv_cache_qformat none \
    --calib_with_images \
    --calib_size 512

/usr/local/lib/python3.12/dist-packages/torch/cuda/__init__.py:61: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]
Error.  nthreads cannot be larger than environment variable "NUMEXPR_MAX_THREADS" (64)/workspace/user_homes/lmikaelyan/Model-Optimizer/modelopt/torch/__init__.py:55: UserWarning: transformers>=5.0 support is experimental. Unified Hugging Face checkpoint export for quantized checkpoints may not work for some models yet.
  _warnings.warn(
ModelOpt save/restore enabled for `transformers` library.
ModelOpt save/restore enabled for `diffusers` library.
ModelOpt save/restore enabled for `peft` library.
Failed to get GPU memory info: Uninitialized. Stopping GPU memory monitor.
Initializing model from /workspace/user_homes/lmikaelyan/Qwen3-VL-8B-Instruct
`torch_dtype` is deprecated!

Key flags:

- `--qformat fp8` — FP8 weights + activations (E4M3). Use `nvfp4` for Blackwell.
- `--kv_cache_qformat none` — leave KV-cache in BF16. FP8 KV-cache is faster but can hurt longer-context accuracy.
- `--calib_with_images` — VLM-specific: build the calibration set from image-text pairs (uses `nemotron_vlm_dataset_v2`).
- `--calib_size 512` — number of calibration samples. 512 is a good default; more is rarely worth it.

Run `python ../../llm_ptq/hf_ptq.py --help` for the full flag list — including `nvfp4`, `int8_sq`, AWQ, and KV-cache quant variants.

---
## 3.3 (Optional) NVFP4 PTQ

On Blackwell, swap `fp8` for `nvfp4`. The command is otherwise identical:

```bash
python ../../llm_ptq/hf_ptq.py \
    --pyt_ckpt_path /workspace/user_homes/lmikaelyan/Qwen3-VL-distilled-smoke \
    --export_path   /workspace/user_homes/lmikaelyan/Qwen3-VL-distilled-nvfp4 \
    --qformat nvfp4 \
    --kv_cache_qformat nvfp4 \
    --calib_with_images \
    --calib_size 512 \
    --trust_remote_code
```

---
## 3.4 Inspect the FP8 export — and how to verify it

ModelOpt's FP8 export uses `quantization_config.quant_method = "modelopt"`. **vLLM understands this natively, but `transformers>=5.0`'s `AutoModelForImageTextToText.from_pretrained` does not** — it silently skips dequantization, treats every `*.weight_scale` / `*.input_scale` tensor as UNEXPECTED, and loads the raw FP8 bit-patterns as if they were BF16. The model then generates gibberish, even though the underlying FP8 checkpoint is correct.

So we **don't** try to load the FP8 export with HF transformers here. Instead:

- **§3.4.a** — print the export's file inventory and size to confirm PTQ produced what we expect.
- **§3.4.b** — for an actual end-to-end functional check on the FP8 weights, **serve the export via vLLM** (notebook [04_serving_and_eval_vlm.ipynb](04_serving_and_eval_vlm.ipynb)). vLLM dequantizes correctly and a single chat-completion request is enough to confirm the FP8 checkpoint is healthy.

In [ ]:
# §3.4.a Inspect the export
import json, os
from pathlib import Path

EXPORT_PATH = "/workspace/user_homes/lmikaelyan/Qwen3-VL-distilled-fp8"

for f in sorted(Path(EXPORT_PATH).iterdir()):
    if not f.is_file():
        continue
    size = os.path.getsize(f)
    fmt = f"{size/1e9:>6.2f} GB" if size > 1e8 else f"{size:>8} B"
    print(f"  {f.name:<40} {fmt}")

cfg = json.load(open(f"{EXPORT_PATH}/config.json"))
qc  = cfg["quantization_config"]
print(f"\nquant_method:  {qc['quant_method']}")
print(f"quant_algo:    {qc['quant_algo']}")
print(f"ignored:       {qc.get('ignore', qc.get('exclude_modules', []))}")

total = sum(os.path.getsize(p) for p in Path(EXPORT_PATH).rglob("*.safetensors"))
print(f"\nFP8 weights on disk: {total/1e9:.2f} GB (vs ~15 GB BF16)")